In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import mpl_lego as mplego

from mpl_lego.style import use_latex_style
from mpl_lego.labels import bold_text, bold_axis_ticklabels
from interviewer.opacity_columns import mechanism_columns, MECHANISM_KEYS, level_column


In [ ]:
use_latex_style()

In [ ]:
PROJECT_ROOT = Path.cwd().parents[0]
RESULTS_PATH = PROJECT_ROOT / "outputs/runs/exp0.3/results.csv"

In [ ]:
results_path = Path(RESULTS_PATH)
df = pd.read_csv(results_path)

In [ ]:
level_order = ["none", "potential", "clear"]
mechanisms = [m for m in MECHANISM_KEYS if level_column(m) in df.columns]

rng = np.random.default_rng(7)
n_bootstrap = 1000
ci_lower = 2.5
ci_upper = 97.5

rows = []
for mechanism in mechanisms:
    values = df[level_column(mechanism)].dropna().to_numpy()
    n_values = len(values)

    bootstrap_fractions = {level: [] for level in level_order}
    if n_values:
        for _ in range(n_bootstrap):
            sample = values[rng.integers(0, n_values, n_values)]
            for level in level_order:
                bootstrap_fractions[level].append(np.mean(sample == level))

    observed = pd.Series(values).value_counts(normalize=True) if n_values else pd.Series(dtype=float)

    for level in level_order:
        fraction = float(observed.get(level, 0.0))
        if bootstrap_fractions[level]:
            lower = float(np.percentile(bootstrap_fractions[level], ci_lower))
            upper = float(np.percentile(bootstrap_fractions[level], ci_upper))
        else:
            lower = 0.0
            upper = 0.0

        rows.append({
            "mechanism": mechanism,
            "level": level,
            "fraction": fraction,
            "ci_lower": lower,
            "ci_upper": upper,
        })

level_fractions = pd.DataFrame(rows)

palette = mplego.colors.get_default_ccycle()
fig, ax = plt.subplots(figsize=(8, 4))

x_positions = np.arange(len(mechanisms))
width = 0.22

for idx, level in enumerate(level_order):
    subset = (
        level_fractions[level_fractions["level"] == level]
        .set_index("mechanism")
        .reindex(mechanisms)
        .fillna({"fraction": 0.0, "ci_lower": 0.0, "ci_upper": 0.0})
    )
    offsets = x_positions + (idx - 1) * width
    heights = subset["fraction"].to_numpy()
    yerr = np.vstack([
        heights - subset["ci_lower"].to_numpy(),
        subset["ci_upper"].to_numpy() - heights,
    ])
    ax.bar(
        offsets,
        heights,
        width=width,
        label=level,
        color=palette[idx % len(palette)],
        yerr=yerr,
        capsize=3,
        ecolor="black",
        linewidth=0,
    )

mechanism_labels = [m.replace("_opacity", "").replace("_", " ").title() for m in mechanisms]
ax.set_xticks(x_positions)
ax.set_xticklabels(mechanism_labels)

xlabel = "Mechanism"
ylabel = "Fraction of samples"
if plt.rcParams.get("text.usetex"):
    xlabel = mplego.labels.bold_text(xlabel)
    ylabel = mplego.labels.bold_text(ylabel)

ax.set_xlabel(xlabel, fontweight="bold")
ax.set_ylabel(ylabel, fontweight="bold")
ax.set_ylim(0, 1)
ax.legend(title="Level")
ax.set_facecolor('whitesmoke')
ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.set_axisbelow(True)
title = "All Interviews"
if plt.rcParams.get("text.usetex"):
    title = mplego.labels.bold_text(title)
ax.set_title(title, fontsize=14, fontweight="bold")
fig.tight_layout()


In [ ]:
subset_definitions = {
    "Work Interviews": df["transcript_id"].astype(str).str.startswith("work_"),
    "Creativity Interviews": df["transcript_id"].astype(str).str.startswith("creativity_"),
    "Science Interviews": df["transcript_id"].astype(str).str.startswith("science_"),
}

level_order = ["none", "potential", "clear"]
mechanisms = [m for m in MECHANISM_KEYS if level_column(m) in df.columns]
palette = mplego.colors.get_default_ccycle()
width = 0.22
n_bootstrap = 1000
ci_lower = 2.5
ci_upper = 97.5

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5), sharey=True)
legend_handles = None
legend_labels = None

for panel_idx, (panel_title, mask) in enumerate(subset_definitions.items()):
    subset_df = df.loc[mask].copy()
    rng = np.random.default_rng(7 + panel_idx)

    rows = []
    for mechanism in mechanisms:
        values = subset_df[level_column(mechanism)].dropna().to_numpy()
        n_values = len(values)

        bootstrap_fractions = {level: [] for level in level_order}
        if n_values:
            for _ in range(n_bootstrap):
                sample = values[rng.integers(0, n_values, n_values)]
                for level in level_order:
                    bootstrap_fractions[level].append(np.mean(sample == level))

        observed = pd.Series(values).value_counts(normalize=True) if n_values else pd.Series(dtype=float)

        for level in level_order:
            fraction = float(observed.get(level, 0.0))
            if bootstrap_fractions[level]:
                lower = float(np.percentile(bootstrap_fractions[level], ci_lower))
                upper = float(np.percentile(bootstrap_fractions[level], ci_upper))
            else:
                lower = 0.0
                upper = 0.0

            rows.append({
                "mechanism": mechanism,
                "level": level,
                "fraction": fraction,
                "ci_lower": lower,
                "ci_upper": upper,
            })

    level_fractions = pd.DataFrame(rows)
    ax = axes[panel_idx]
    x_positions = np.arange(len(mechanisms))

    for idx, level in enumerate(level_order):
        chart_data = (
            level_fractions[level_fractions["level"] == level]
            .set_index("mechanism")
            .reindex(mechanisms)
            .fillna({"fraction": 0.0, "ci_lower": 0.0, "ci_upper": 0.0})
        )
        offsets = x_positions + (idx - 1) * width
        heights = chart_data["fraction"].to_numpy()
        yerr = np.vstack([
            heights - chart_data["ci_lower"].to_numpy(),
            chart_data["ci_upper"].to_numpy() - heights,
        ])
        bars = ax.bar(
            offsets,
            heights,
            width=width,
            label=level,
            color=palette[idx % len(palette)],
            yerr=yerr,
            capsize=3,
            ecolor="black",
            linewidth=0,
        )
        if panel_idx == 0:
            if legend_handles is None:
                legend_handles = []
                legend_labels = []
            legend_handles.append(bars)
            legend_labels.append(level)

    mechanism_labels = [m.replace("_opacity", "").replace("_", " ").title() for m in mechanisms]
    ax.set_xticks(x_positions)
    ax.set_xticklabels(mechanism_labels)

    xlabel = "Mechanism"
    if plt.rcParams.get("text.usetex"):
        xlabel = mplego.labels.bold_text(xlabel)
    ax.set_xlabel(xlabel, fontweight="bold")

    if panel_idx == 0:
        ylabel = "Fraction of samples"
        if plt.rcParams.get("text.usetex"):
            ylabel = mplego.labels.bold_text(ylabel)
        ax.set_ylabel(ylabel, fontweight="bold")

    title = panel_title
    if plt.rcParams.get("text.usetex"):
        title = mplego.labels.bold_text(title)
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_ylim(0, 1)
    ax.set_facecolor("whitesmoke")
    ax.grid(axis="y", linestyle="--", alpha=0.7)
    ax.set_axisbelow(True)

fig.legend(
    legend_handles,
    legend_labels,
    loc="lower center",
    ncol=3,
    frameon=True,
    bbox_to_anchor=(0.5, -0.03),
    title=bold_text("Level"),
)
fig.tight_layout(rect=(0, 0.08, 1, 1))


In [ ]:
subset_definitions = {
    "Work Interviews": df["transcript_id"].astype(str).str.startswith("work_"),
    "Creativity Interviews": df["transcript_id"].astype(str).str.startswith("creativity_"),
    "Science Interviews": df["transcript_id"].astype(str).str.startswith("science_"),
}

form_order = ["avoidance", "mixed", "production"]
mechanisms = [m for m in MECHANISM_KEYS if level_column(m) in df.columns]
palette = mplego.colors.get_default_ccycle()
width = 0.22
n_bootstrap = 1000
ci_lower = 2.5
ci_upper = 97.5

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5), sharey=True)
legend_handles = []
legend_labels = []

for panel_idx, (panel_title, mask) in enumerate(subset_definitions.items()):
    subset_df = df.loc[mask].copy()
    rng = np.random.default_rng(17 + panel_idx)

    rows = []
    for mechanism in mechanisms:
        level_col = f"{mechanism}_level"
        form_col = f"{mechanism}_form"
        conditioned = subset_df.loc[subset_df[level_col].ne("none"), form_col].dropna().to_numpy()
        conditioned = conditioned[conditioned != "none"]
        n_values = len(conditioned)

        bootstrap_fractions = {form: [] for form in form_order}
        if n_values:
            for _ in range(n_bootstrap):
                sample = conditioned[rng.integers(0, n_values, n_values)]
                for form in form_order:
                    bootstrap_fractions[form].append(np.mean(sample == form))

        observed = pd.Series(conditioned).value_counts(normalize=True) if n_values else pd.Series(dtype=float)

        for form in form_order:
            fraction = float(observed.get(form, 0.0))
            if bootstrap_fractions[form]:
                lower = float(np.percentile(bootstrap_fractions[form], ci_lower))
                upper = float(np.percentile(bootstrap_fractions[form], ci_upper))
            else:
                lower = 0.0
                upper = 0.0

            rows.append({
                "mechanism": mechanism,
                "form": form,
                "fraction": fraction,
                "ci_lower": lower,
                "ci_upper": upper,
            })

    form_fractions = pd.DataFrame(rows)
    ax = axes[panel_idx]
    x_positions = np.arange(len(mechanisms))

    for idx, form in enumerate(form_order):
        chart_data = (
            form_fractions[form_fractions["form"] == form]
            .set_index("mechanism")
            .reindex(mechanisms)
            .fillna({"fraction": 0.0, "ci_lower": 0.0, "ci_upper": 0.0})
        )
        offsets = x_positions + (idx - 1) * width
        heights = chart_data["fraction"].to_numpy()
        yerr = np.vstack([
            heights - chart_data["ci_lower"].to_numpy(),
            chart_data["ci_upper"].to_numpy() - heights,
        ])
        bars = ax.bar(
            offsets,
            heights,
            width=width,
            label=form,
            color=palette[idx % len(palette)],
            yerr=yerr,
            capsize=3,
            ecolor="black",
            linewidth=0,
        )
        if panel_idx == 0:
            legend_handles.append(bars)
            legend_labels.append(form)

    mechanism_labels = [m.replace("_opacity", "").replace("_", " ").title() for m in mechanisms]
    ax.set_xticks(x_positions)
    ax.set_xticklabels(mechanism_labels)

    xlabel = "Mechanism"
    if plt.rcParams.get("text.usetex"):
        xlabel = mplego.labels.bold_text(xlabel)
    ax.set_xlabel(xlabel, fontweight="bold")

    if panel_idx == 0:
        ylabel = "Fraction of Cases with Mechanism Evidence"
        if plt.rcParams.get("text.usetex"):
            ylabel = mplego.labels.bold_text(ylabel)
        ax.set_ylabel(ylabel, fontweight="bold")

    title = panel_title
    if plt.rcParams.get("text.usetex"):
        title = mplego.labels.bold_text(title)
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_ylim(0, 1)
    ax.set_facecolor("whitesmoke")
    ax.grid(axis="y", linestyle="--", alpha=0.7)
    ax.set_axisbelow(True)

legend_title = "Form"
if plt.rcParams.get("text.usetex"):
    legend_title = mplego.labels.bold_text(legend_title)

fig.legend(
    legend_handles,
    legend_labels,
    loc="lower center",
    ncol=3,
    frameon=False,
    bbox_to_anchor=(0.5, -0.03),
    title=legend_title,
)
fig.tight_layout(rect=(0, 0.08, 1, 1))
